# CommercialRentAI Colab 학습

Google Drive의 `내 드라이브 > Colab Notebooks > 상가 임대료 AI` 폴더에 있는 데이터를 Colab 로컬 디스크(`/content`)로 복사한 뒤 학습합니다.

Drive에서 CSV를 직접 오래 읽으면 `Transport endpoint is not connected` 오류가 날 수 있으므로, 학습 중 대량 읽기는 로컬 디스크에서 처리합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/상가 임대료 AI')
LOCAL_DATA_DIR = Path('/content/commercial_rent_ai_data')
LOCAL_OUTPUT_DIR = Path('/content/commercial_rent_ai_output')

print('PROJECT_DIR:', PROJECT_DIR)
print('LOCAL_DATA_DIR:', LOCAL_DATA_DIR)
print('LOCAL_OUTPUT_DIR:', LOCAL_OUTPUT_DIR)
print('train script exists:', (PROJECT_DIR / 'train_commercial_rent.py').exists())
print('requirements exists:', (PROJECT_DIR / 'requirements.txt').exists())

`train_commercial_rent.py`가 없다고 나오면, 로컬 프로젝트의 `AI/CommercialRentAI` 폴더 안 파일들을 Google Drive의 `Colab Notebooks/상가 임대료 AI` 폴더에 올린 뒤 다시 실행하세요.

In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/상가 임대료 AI"
from pathlib import Path
import sys

if Path('requirements.txt').exists():
    !{sys.executable} -m pip install -r requirements.txt
else:
    !{sys.executable} -m pip install pandas numpy joblib scikit-learn lightgbm openpyxl

Drive의 데이터 파일을 Colab 로컬 디스크로 복사합니다. 이전 실행 결과물인 `models/`, `config/`, zip 파일은 복사하지 않습니다.

In [ ]:
import shutil

if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

skip_names = {
    'models',
    'config',
    '__pycache__',
    'commercial_rent_ai_outputs.zip',
    'commercial_rent_ai_output.zip',
}
skip_suffixes = {'.ipynb', '.gdoc', '.gsheet', '.gslides', '.shortcut-targets-by-id'}

for src in PROJECT_DIR.iterdir():
    if src.name in skip_names or src.suffix.lower() in skip_suffixes:
        continue
    dst = LOCAL_DATA_DIR / src.name
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

print('Copied files to:', LOCAL_DATA_DIR)
print('Top-level items:', sorted(p.name for p in LOCAL_DATA_DIR.iterdir()))

In [ ]:
%cd /content/commercial_rent_ai_data
!python train_commercial_rent.py \
  --data-dir "/content/commercial_rent_ai_data" \
  --output-dir "/content/commercial_rent_ai_output" \
  --device auto \
  --zip-output

학습 결과만 Drive 폴더로 복사합니다.

In [ ]:
import shutil

for name in ['models', 'config']:
    src = LOCAL_OUTPUT_DIR / name
    dst = PROJECT_DIR / name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

zip_src = LOCAL_OUTPUT_DIR / 'commercial_rent_ai_outputs.zip'
zip_dst = PROJECT_DIR / 'commercial_rent_ai_outputs.zip'
if zip_src.exists():
    shutil.copy2(zip_src, zip_dst)

print('Copied training outputs to:', PROJECT_DIR)

In [ ]:
for path in [
    PROJECT_DIR / 'models/rent_h1m.pkl',
    PROJECT_DIR / 'models/rent_h3m.pkl',
    PROJECT_DIR / 'models/rent_h6m.pkl',
    PROJECT_DIR / 'config/model_meta.json',
    PROJECT_DIR / 'commercial_rent_ai_outputs.zip',
]:
    print(path.name, 'OK' if path.exists() else 'MISSING')

생성된 `models/`와 `config/` 폴더를 로컬 `D:\dev\JAVA_project\ZIPHYEONJEON\AI\CommercialRentAI` 아래에 복사한 뒤 `python main.py`를 실행하면 API가 학습된 모델을 로드합니다.